# Overfitting and Underfitting

**Topic:** Generalization vs Memorization

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as widgets
from ipywidgets import IntSlider, FloatSlider, Dropdown, ToggleButtons, Output, HBox, VBox, Label
from IPython.display import display, clear_output
from scipy import stats
from sklearn.datasets import make_classification, make_regression, make_circles, make_moons
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** the symptoms of overfitting and underfitting by looking at training vs validation error
- **Explain** why a model with very low training error can still fail on new data
- **Interpret** how regularization pulls a diverging validation curve back toward the training curve

> **Tip:** Drag the **Complexity slider** from 1 to 15 and watch the validation error. Then switch on **Regularization** and repeat the experiment. Notice how regularization changes where the curves diverge.

---
## How we got here

In **04_bias_variance_tradeoff.ipynb** we decomposed error into bias and variance. Overfitting and underfitting are the practical names for those two failure modes:

- **Underfitting** = high bias — the model is too simple to capture the pattern
- **Overfitting** = high variance — the model memorizes the training data rather than learning its structure

Here we make the abstract theory concrete by watching it happen on real learning curves.

---
## Why this matters for data science

Overfitting is the single most common failure mode in machine learning. A model that memorizes the training set will look spectacular on training data and fail on everything else — which is exactly the data that matters in production.

Recognizing overfitting from a learning curve, and knowing which tools (regularization, more data, simpler architecture, cross-validation) can fix it, is a core practical skill for any data scientist.

---
## Try it yourself

In [2]:
np.random.seed(42)
X, y = make_classification(n_samples=300, n_features=2,
    n_informative=2, n_redundant=0, random_state=42)

out = Output()

complexity_slider = IntSlider(value=1, min=1, max=15, step=1,
    description="Complexity:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"))

reg_toggle = ToggleButtons(
    options=["No regularization", "With regularization"],
    value="No regularization",
    description="",
    layout=widgets.Layout(width="420px"))

def complexity_curves(regularize):
    from sklearn.model_selection import validation_curve
    depths = list(range(1, 16))
    base = (DecisionTreeClassifier(min_samples_leaf=5, random_state=42)
            if regularize else DecisionTreeClassifier(random_state=42))
    train_scores, val_scores = validation_curve(
        base, X, y, param_name="max_depth", param_range=depths,
        cv=5, scoring="neg_mean_squared_error")
    return depths, (-train_scores.mean(axis=1)).tolist(), (-val_scores.mean(axis=1)).tolist()

depths, train_e, val_e = complexity_curves(False)
depths, train_e_reg, val_e_reg = complexity_curves(True)

def render(depth, regularize):
    te = train_e_reg if regularize else train_e
    ve = val_e_reg   if regularize else val_e
    diag = ("Underfitting" if depth <= 2
            else ("Overfitting" if depth >= 10 and not regularize else "Good fit"))
    traces = [
        go.Scatter(x=depths, y=te, mode="lines+markers",
            line=dict(color=PALETTE["accent"], width=2), name="Training error"),
        go.Scatter(x=depths, y=ve, mode="lines+markers",
            line=dict(color=PALETTE["secondary"], width=2), name="Validation error"),
        go.Scatter(x=[depth], y=[ve[depth-1]], mode="markers",
            marker=dict(color=PALETTE["primary"], size=14), name="Current"),
    ]
    layout = base_layout(
        title=f"Depth {depth} — {diag}{'  (regularized)' if regularize else ''}",
        xaxis_title="Model complexity (max tree depth)", yaxis_title="Error")
    with out:
        clear_output(wait=True)
        display(go.FigureWidget(go.Figure(data=traces, layout=layout)))

def on_change(change):
    render(complexity_slider.value, reg_toggle.value == "With regularization")

complexity_slider.observe(on_change, names="value")
reg_toggle.observe(on_change, names="value")
display(VBox([complexity_slider, reg_toggle, out]))
render(1, False)

---
## What's happening?

Think of a student preparing for an exam:

- **Underfitting** is the student who barely studied. They get poor scores on both practice tests and the real exam. The model simply does not know enough.
- **Overfitting** is the student who memorized every past exam question word-for-word. They ace practice tests but fail when the real exam uses different phrasing. The model has memorized, not understood.
- **Good fit** is the student who understood the underlying concepts. Practice scores and exam scores are close.

In the charts above:
- Underfitting: both training and validation error are high
- Good fit: both curves are low and close together
- Overfitting: training error is near zero, validation error diverges upward

| Symptom | Training error | Validation error | Diagnosis | Fix |
|---|---|---|---|---|
| Both high | High | High | Underfitting | More complexity, more features |
| Both low | Low | Low | Good fit | Ship it |
| Gap widening | Low | Rising | Overfitting | Regularize, simplify, get more data |

---
## Real-world example: Credit risk model

A bank trains a gradient boosting model with 500 trees on 50,000 loan applications. Training accuracy: 99.8%. The model memorized edge cases in the historical data, including economic conditions that no longer apply.

Validation on the last 6 months of data shows 76% accuracy. The model overfit to historical patterns. Applying regularization (fewer trees, max depth constraint) reduces training accuracy to 91% but raises validation accuracy to 89% — a far more reliable model for production.

---
## Key takeaway

> **Overfitting means the model memorized the training data rather than learning its pattern — and a validation curve that rises while training error falls is the diagnostic signal.**

---
*Next up: Cross-Validation — the tool that gives you a reliable estimate of validation error*